# Imputation Technique 2: Constant Imputation

**Dataset:** `Loan_Default.csv`

**When to use:** When missing data is believed to be **Missing Not At Random (MNAR)** — i.e., the absence itself carries meaning. A sentinel value (like `0` or `-1`) signals to the model that this value was deliberately absent.

---


### Step 1: Setup — Data Loading & Prep


In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

df_freq = df.copy()
for c in Nominal_features:
    df_freq[c + '_freq'] = df_freq[c].map(df_freq.groupby(c).size() / df_freq.shape[0])
    indexer = pd.factorize(df_freq[c], sort=True)[1]
    df_freq[c] = indexer.get_indexer(df_freq[c])
df_freq = df_freq.drop(Nominal_features, axis=1)

# Drop high-missing columns first
high_missing_cols = ['rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'property_value', 'LTV', 'dtir1']
df_drop = df_freq.drop(high_missing_cols, axis=1)

print(f'Starting shape: {df_drop.shape}')

Starting shape: (148670, 26)


### Step 2: Define Columns to Impute


In [2]:
Cols_to_be_imputed = [
    'term', 'income', 'age',
    'loan_limit_freq', 'approv_in_adv_freq', 'loan_purpose_freq',
    'Neg_ammortization_freq', 'submission_of_application_freq'
]

### Step 3: Apply Constant Imputation

`strategy='constant'` fills all NaNs with a fixed value (default is `0`).


In [3]:
df_const = df_drop.copy()

Const_imputer = SimpleImputer(strategy='constant') # Defaults to 0
df_const[Cols_to_be_imputed] = Const_imputer.fit_transform(df_const[Cols_to_be_imputed])

df_const.head()

,loan_amount,term,income,Credit_Score,age,Status,loan_limit_freq,Gender_freq,approv_in_adv_freq,loan_type_freq,...,lump_sum_payment_freq,construction_type_freq,occupancy_type_freq,Secured_by_freq,total_units_freq,credit_type_freq,co-applicant_credit_type_freq,submission_of_application_freq,Region_freq,Security_Type_freq
0,116500,360.0,1740.0,758,0.0,1,0.910392,0.253306,0.838239,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.644474,0.430591,0.999778
1,206500,360.0,4980.0,552,3.0,1,0.910392,0.284832,0.838239,0.139652,...,0.022762,0.999778,0.929582,0.999778,0.985269,0.102899,0.499617,0.644474,0.502603,0.999778
2,406500,360.0,9480.0,834,1.0,0,0.910392,0.284832,0.155653,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.644474,0.430591,0.999778
3,456500,360.0,11880.0,587,2.0,0,0.910392,0.284832,0.838239,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.277924,0.500383,0.354180,0.502603,0.999778
4,696500,360.0,10440.0,602,0.0,0,0.910392,0.278462,0.155653,0.761236,...,0.977238,0.999778,0.929582,0.999778,0.985269,0.295292,0.499617,0.354180,0.502603,0.999778


### Step 4: Verify Results


In [4]:
missing_after = df_const.isna().sum()
print('Missing values after Constant Imputation:')
print(missing_after[missing_after > 0] if missing_after.sum() > 0 else 'None — all filled!')

Missing values after Constant Imputation:
None — all filled!
